# RGB Benchmark Evaluation Notebook (FIXED)
## Benchmarking LLM Abilities for Retrieval-Augmented Generation

**Paper:** [Benchmarking Large Language Models in Retrieval-Augmented Generation](https://arxiv.org/pdf/2309.01431)
**Dataset:** [RGB GitHub Repository](https://github.com/chen700564/RGB)

---

### What This Notebook Does
RGB tests **4 fundamental LLM abilities** required for reliable RAG systems:

| # | Ability | What It Tests | Metric |
|---|---------|--------------|--------|
| 1 | **Noise Robustness** | Can the LLM extract correct answers despite noisy/irrelevant docs? | Accuracy |
| 2 | **Negative Rejection** | Can the LLM refuse to answer when no relevant doc exists? | Rejection Rate |
| 3 | **Information Integration** | Can the LLM combine info from multiple docs? | Accuracy |
| 4 | **Counterfactual Robustness** | Can the LLM detect & correct factual errors in docs? | Error Detection + Correction Rate |

> **Key difference from RAGBench:** RGB does NOT require you to build a retriever. Documents are pre-provided per question. You only evaluate the LLM's generation quality.

### Fixes Applied in This Version
1. **Document extraction** now correctly handles RGB's actual data keys (`positive`, `negative`, `positive_wrong`)
2. **Noise Robustness** correctly passes positive + negative documents to the LLM
3. **Negative Rejection** correctly uses ONLY `negative` documents (irrelevant), so LLM should refuse
4. **Information Integration** correctly flattens nested positive doc lists and combines with negatives
5. **Counterfactual Robustness** correctly uses `positive_wrong` documents (docs with deliberate factual errors)
6. **Integration accuracy** now checks that ALL answer components are found (not just any one)

## Step 1: Environment Setup
Install required packages and set up Groq API access.

In [ ]:
# Install required packages
!pip install -q groq pandas matplotlib seaborn tqdm

In [ ]:
import json
import os
import re
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from groq import Groq
from tqdm import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print('All imports successful!')

In [ ]:
# ============================================================
# CONFIGURATION - Set your Groq API key here
# ============================================================

from google.colab import userdata, files as colab_files

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

# Models to evaluate - add/remove as needed
MODELS_TO_EVALUATE = [
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b",
    # "llama-3.3-70b-versatile",   # Uncomment if you have quota
    # "deepseek-r1-distill-llama-70b",
]

print(f'Groq client initialized. Will evaluate {len(MODELS_TO_EVALUATE)} models.')

## Step 2: Download the RGB Dataset
The RGB dataset lives on GitHub (NOT HuggingFace). It has 3 JSON files for English:
- `en.json` → Noise Robustness (300 questions) + Negative Rejection (uses same questions but with all-negative docs)
- `en_int.json` → Information Integration (100 questions requiring multi-doc reasoning)
- `en_fact.json` → Counterfactual Robustness (100 questions with deliberately wrong facts in docs)

In [ ]:
# Download RGB dataset from GitHub
import urllib.request

os.makedirs('rgb_data', exist_ok=True)

# RGB dataset URLs from the official repo
RGB_FILES = {
    'en': 'https://raw.githubusercontent.com/chen700564/RGB/refs/heads/master/data/en.json',
    'en_int': 'https://raw.githubusercontent.com/chen700564/RGB/refs/heads/master/data/en_int.json',
    'en_fact': 'https://raw.githubusercontent.com/chen700564/RGB/refs/heads/master/data/en_fact.json',
}

for name, url in RGB_FILES.items():
    filepath = f'rgb_data/{name}.json'
    if not os.path.exists(filepath):
        print(f'Downloading {name}.json ...')
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f'  ✅ Saved to {filepath}')
        except Exception as e:
            print(f'  ❌ Error downloading {name}: {e}')
            print(f'  💡 Manually download from: {url}')
    else:
        print(f'  ✅ {filepath} already exists')

print('\nDone! Check rgb_data/ folder.')

## Step 3: Load & Explore the Dataset
Understand the structure of each JSON file before writing evaluation code.

In [ ]:
# Load all three datasets
def load_rgb_data(filepath):
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data

data_en = load_rgb_data('rgb_data/en.json')
data_int = load_rgb_data('rgb_data/en_int.json')
data_fact = load_rgb_data('rgb_data/en_fact.json')

print(f'en.json      : {len(data_en)} questions  (Noise Robustness + Negative Rejection)')
print(f'en_int.json  : {len(data_int)} questions  (Information Integration)')
print(f'en_fact.json : {len(data_fact)} questions  (Counterfactual Robustness)')
print(f'\nTotal English questions: {len(data_en) + len(data_int) + len(data_fact)}')

In [ ]:
# Explore the structure of one sample from each dataset
print('=' * 60)
print('SAMPLE FROM en.json (Noise Robustness / Negative Rejection)')
print('=' * 60)
sample = data_en[0]
print(f'Keys: {list(sample.keys())}')
for k, v in sample.items():
    val_str = str(v)[:200]
    print(f'  {k}: {type(v).__name__} = {val_str}')
print(f'  Number of positive docs: {len(sample.get("positive", []))}')
print(f'  Number of negative docs: {len(sample.get("negative", []))}')

print('\n' + '=' * 60)
print('SAMPLE FROM en_int.json (Information Integration)')
print('=' * 60)
sample_int = data_int[0]
for k, v in sample_int.items():
    val_str = str(v)[:200]
    print(f'  {k}: {type(v).__name__} = {val_str}')
# Check if positive is nested
if 'positive' in sample_int and sample_int['positive']:
    first_pos = sample_int['positive'][0]
    print(f'  positive[0] type: {type(first_pos).__name__}')
    if isinstance(first_pos, list):
        print(f'  positive is a NESTED list (list of doc groups) — each group has {len(first_pos)} docs')

print('\n' + '=' * 60)
print('SAMPLE FROM en_fact.json (Counterfactual Robustness)')
print('=' * 60)
sample_fact = data_fact[0]
for k, v in sample_fact.items():
    val_str = str(v)[:200]
    print(f'  {k}: {type(v).__name__} = {val_str}')
print(f'  Number of positive_wrong docs: {len(sample_fact.get("positive_wrong", []))}')
print(f'  Number of positive docs: {len(sample_fact.get("positive", []))}')
print(f'  Number of negative docs: {len(sample_fact.get("negative", []))}')

## Step 4: Define Helper Functions
### 4a. LLM Call Wrapper with Rate-Limit Handling
Groq has rate limits — we need retry logic with exponential backoff.

In [ ]:
def call_llm(model_name, system_prompt, user_prompt, max_retries=5, temperature=0.1):
    """
    Call Groq LLM with retry logic for rate limits.
    Returns the generated text or None on failure.
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=temperature,
                max_tokens=1024,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            error_str = str(e)
            if 'rate_limit' in error_str.lower() or '429' in error_str:
                wait_time = 2 ** attempt * 10  # 10, 20, 40, 80, 160 seconds
                print(f'    ⏳ Rate limited. Waiting {wait_time}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait_time)
            else:
                print(f'    ❌ Error: {error_str[:200]}')
                return None
    print(f'    ❌ Max retries exceeded for model {model_name}')
    return None

# Quick test
test_resp = call_llm(MODELS_TO_EVALUATE[0], 'You are helpful.', 'Say hello in 5 words.')
print(f'Test response: {test_resp}')

### 4b. Prompt Templates for Each RGB Ability
Each ability needs a specific prompt structure. These follow the RGB paper's approach.

In [ ]:
# ============================================================
# PROMPT TEMPLATES — Based on RGB paper Section 3
# ============================================================

def build_document_context(documents):
    """
    Build a formatted context string from the list of documents.
    Handles both string lists and dict lists.
    """
    context_parts = []
    for i, doc in enumerate(documents, 1):
        if isinstance(doc, dict):
            text = doc.get('text', doc.get('content', doc.get('passage', str(doc))))
        else:
            text = str(doc)
        context_parts.append(f'Document [{i}]: {text}')
    return '\n\n'.join(context_parts)


# --- Noise Robustness & Information Integration ---
SYSTEM_PROMPT_ACCURACY = """You are a helpful assistant. Answer the question based on the given documents.
Provide a concise and direct answer. If the answer involves a specific name, date, number, or entity, state it clearly."""

USER_PROMPT_ACCURACY = """Given the following documents, answer the question.

{context}

Question: {question}

Answer:"""


# --- Negative Rejection ---
SYSTEM_PROMPT_REJECTION = """You are a helpful assistant. Answer the question based ONLY on the given documents.
If the provided documents do not contain sufficient information to answer the question,
you MUST respond with exactly: "I can not answer the question because of insufficient information in documents."
Do not guess or make up an answer."""

USER_PROMPT_REJECTION = """Given the following documents, answer the question.
If the documents do not contain enough information, respond with:
"I can not answer the question because of insufficient information in documents."

{context}

Question: {question}

Answer:"""


# --- Counterfactual Robustness ---
SYSTEM_PROMPT_COUNTERFACTUAL = """You are a helpful assistant. Answer the question based on the given documents.
IMPORTANT: Some documents may contain factual errors or incorrect information.
If you detect factual errors in any document, first state:
"There are factual errors in the provided documents."
Then provide the correct answer based on your knowledge."""

USER_PROMPT_COUNTERFACTUAL = """Given the following documents, answer the question.
WARNING: Some documents may contain factual errors.
If you find factual errors, first say "There are factual errors in the provided documents."
Then give the correct answer.

{context}

Question: {question}

Answer:"""

print('✅ All prompt templates defined.')

### 4c. Data Extraction Helpers (FIXED)
Extract questions, documents, and answers from the RGB JSON structure.

**Key fix:** The RGB dataset uses `positive` and `negative` as document keys (not `documents` or `passages`).
Each evaluation type needs different document combinations:
- **Noise Robustness**: `positive` + `negative` (mix of relevant and irrelevant)
- **Negative Rejection**: `negative` only (all irrelevant — LLM should refuse)
- **Information Integration**: flattened `positive` + `negative` (positive is nested: list of lists)
- **Counterfactual Robustness**: `positive_wrong` (documents with deliberately wrong facts)

In [ ]:
def extract_question(item):
    """Extract question text from an RGB data item."""
    for key in ['question', 'query', 'input']:
        if key in item:
            return item[key]
    return None

def extract_answer(item):
    """Extract ground truth answer from an RGB data item."""
    for key in ['answer', 'answers', 'gold_answer', 'target']:
        if key in item:
            val = item[key]
            if isinstance(val, list):
                return val  # Some questions have multiple valid answers
            return [val]
    return []


# ============================================================
# FIXED: Document extraction for each evaluation type
# ============================================================
# The RGB dataset uses specific keys:
#   en.json:      'positive' (relevant docs), 'negative' (irrelevant docs)
#   en_int.json:  'positive' (NESTED list of doc groups), 'negative'
#   en_fact.json: 'positive_wrong' (docs with factual errors), 'positive', 'negative'

def get_noise_robustness_docs(item):
    """
    For Noise Robustness: combine positive (relevant) + negative (noisy) documents.
    The LLM must find the correct answer despite noisy irrelevant docs.
    """
    positive = item.get('positive', [])
    negative = item.get('negative', [])
    return positive + negative


def get_negative_rejection_docs(item):
    """
    For Negative Rejection: use ONLY negative (irrelevant) documents.
    The LLM should refuse to answer since no relevant docs are provided.
    """
    return item.get('negative', [])


def get_integration_docs(item):
    """
    For Information Integration: flatten nested positive docs + negative docs.
    In en_int.json, 'positive' is a list of lists (each sub-list is a group of
    docs related to one part of the multi-part answer).
    """
    positive = item.get('positive', [])
    negative = item.get('negative', [])
    
    # Flatten positive docs if they are nested (list of lists)
    flat_positive = []
    for p in positive:
        if isinstance(p, list):
            flat_positive.extend(p)
        else:
            flat_positive.append(p)
    
    return flat_positive + negative


def get_counterfactual_docs(item):
    """
    For Counterfactual Robustness: use 'positive_wrong' documents.
    These are docs that contain deliberately wrong/altered facts.
    The LLM should detect the errors and provide the correct answer.
    """
    return item.get('positive_wrong', [])


# Quick validation
sample = data_en[0]
print(f'Question: {extract_question(sample)[:100]}...')
print(f'Answer: {extract_answer(sample)}')
print(f'Noise Robustness docs: {len(get_noise_robustness_docs(sample))} (positive + negative)')
print(f'Negative Rejection docs: {len(get_negative_rejection_docs(sample))} (negative only)')

sample_int = data_int[0]
print(f'\nIntegration docs: {len(get_integration_docs(sample_int))} (flattened positive + negative)')

sample_fact = data_fact[0]
print(f'Counterfactual docs: {len(get_counterfactual_docs(sample_fact))} (positive_wrong)')

### 4d. Evaluation Metric Functions (FIXED)
Each ability has its own metric:
- **Noise Robustness → Accuracy** (does response contain the GT answer?)
- **Negative Rejection → Rejection Rate** (did the LLM output the rejection phrase?)
- **Information Integration → Accuracy** (ALL answer parts must be found)
- **Counterfactual Robustness → Error Detection Rate + Error Correction Rate**

**Key fix:** `check_integration_accuracy` now verifies that ALL answer components are found,
not just any single one. This is critical for questions like "Who directed X and when was it released?"
where BOTH the director AND the date must appear in the response.

In [ ]:
def check_accuracy(response, ground_truth_answers):
    """
    Accuracy check for Noise Robustness.
    Uses exact matching: if the generated text contains ANY of the ground truth
    answer variants, it is considered correct. (Per RGB paper Section 3.2)

    Args:
        response: LLM generated text
        ground_truth_answers: list of acceptable answers (may contain nested lists)
    Returns:
        1 if correct, 0 if incorrect
    """
    if response is None:
        return 0
    response_lower = response.lower().strip()
    for ans in ground_truth_answers:
        if isinstance(ans, list):
            # Handle nested answer lists — match any variant in the group
            for a in ans:
                if str(a).lower().strip() in response_lower:
                    return 1
        else:
            if str(ans).lower().strip() in response_lower:
                return 1
    return 0


def check_integration_accuracy(response, ground_truth_answers):
    """
    FIXED: Accuracy check for Information Integration.
    For integration questions, the answer has MULTIPLE components (e.g., a director
    AND a release date). ALL components must be found in the response.

    The answer structure in en_int.json looks like:
      [["Jan 2 2022", "January 2, 2022", ...], "Frank Marshall"]
    Each element is an answer component. If it's a list, any variant within matches.
    If it's a string, that exact string must be found.

    Returns:
        1 if ALL components found, 0 otherwise
    """
    if response is None:
        return 0
    response_lower = response.lower().strip()
    
    for ans_component in ground_truth_answers:
        if isinstance(ans_component, list):
            # This component has multiple valid variants — match any one
            found = False
            for variant in ans_component:
                if str(variant).lower().strip() in response_lower:
                    found = True
                    break
            if not found:
                return 0  # This component not found → fail
        else:
            # Single string answer component — must match
            if str(ans_component).lower().strip() not in response_lower:
                return 0  # Not found → fail
    return 1  # All components found


def check_rejection(response):
    """
    Rejection check for Negative Rejection.
    The LLM should output a phrase indicating it cannot answer.
    (Per RGB paper: "I can not answer the question because of insufficient information")

    Returns:
        1 if rejected (good), 0 if it attempted to answer (bad)
    """
    if response is None:
        return 0
    response_lower = response.lower()
    rejection_phrases = [
        'i can not answer',
        'i cannot answer',
        'cannot be answered',
        'insufficient information',
        'not enough information',
        'no relevant information',
        'unable to answer',
        'do not have enough',
        "don't have enough",
        'no information available',
        'documents do not contain',
        "documents don't contain",
        'not mentioned in',
        'no answer can be found',
    ]
    return 1 if any(phrase in response_lower for phrase in rejection_phrases) else 0


def check_error_detection(response):
    """
    Error Detection check for Counterfactual Robustness.
    The LLM should flag that documents contain factual errors.

    Returns:
        1 if error detected, 0 otherwise
    """
    if response is None:
        return 0
    response_lower = response.lower()
    detection_phrases = [
        'factual error',
        'factually incorrect',
        'incorrect information',
        'inaccurate information',
        'error in the',
        'errors in the',
        'not accurate',
        'is incorrect',
        'are incorrect',
        'contains an error',
        'contain errors',
        'factual inaccuracy',
        'misinformation',
        'wrong information',
    ]
    return 1 if any(phrase in response_lower for phrase in detection_phrases) else 0


def check_error_correction(response, ground_truth_answers):
    """
    Error Correction check for Counterfactual Robustness.
    After detecting the error, the LLM should still provide the CORRECT answer.

    Returns:
        1 if correct answer given despite error, 0 otherwise
    """
    return check_accuracy(response, ground_truth_answers)


print('✅ All metric functions defined.')
print()
# Quick tests
print('Test accuracy:', check_accuracy('The answer is Paris', ['Paris']))  # Should be 1
print('Test rejection:', check_rejection('I cannot answer the question'))  # Should be 1
print('Test detection:', check_error_detection('There are factual errors in the documents'))  # Should be 1
print('Test integration (both found):', check_integration_accuracy(
    'Frank Marshall directed it, premiering January 2, 2022',
    [['January 2, 2022', 'Jan 2, 2022'], 'Frank Marshall']
))  # Should be 1
print('Test integration (one missing):', check_integration_accuracy(
    'It premiered on January 2, 2022',
    [['January 2, 2022', 'Jan 2, 2022'], 'Frank Marshall']
))  # Should be 0

## Step 5: Core Evaluation Functions (FIXED)
One function per ability. Each loops through the dataset, calls the LLM, and computes the metric.

**Key fixes:**
- Each function now uses the correct document extraction helper
- Negative Rejection uses only negative docs (instead of broken filtering)
- Information Integration uses `check_integration_accuracy` and flattened docs
- Counterfactual Robustness uses `positive_wrong` docs

In [ ]:
def evaluate_noise_robustness(model_name, data, max_samples=None, delay=1.5):
    """
    FIXED: Evaluate Noise Robustness: Can the LLM answer correctly with noisy docs?
    Uses positive + negative documents from en.json.
    """
    results = []
    samples = data[:max_samples] if max_samples else data

    print(f'\n📊 Noise Robustness — {model_name} — {len(samples)} samples')
    for i, item in enumerate(tqdm(samples, desc='Noise Robustness')):
        question = extract_question(item)
        gt_answers = extract_answer(item)
        documents = get_noise_robustness_docs(item)  # FIXED: positive + negative

        if not question or not gt_answers or not documents:
            continue

        context = build_document_context(documents)
        user_prompt = USER_PROMPT_ACCURACY.format(context=context, question=question)

        response = call_llm(model_name, SYSTEM_PROMPT_ACCURACY, user_prompt)
        correct = check_accuracy(response, gt_answers)

        results.append({
            'index': i,
            'question': question[:100],
            'gt_answer': str(gt_answers),
            'response': response[:300] if response else 'FAILED',
            'correct': correct,
            'num_docs': len(documents),
        })
        time.sleep(delay)  # Respect rate limits

    accuracy = sum(r['correct'] for r in results) / len(results) if results else 0
    print(f'  ✅ Accuracy: {accuracy:.4f} ({sum(r["correct"] for r in results)}/{len(results)})')
    return results, accuracy

print('✅ evaluate_noise_robustness defined')

In [ ]:
def evaluate_negative_rejection(model_name, data, max_samples=None, delay=1.5):
    """
    FIXED: Evaluate Negative Rejection: Can the LLM refuse when docs are all irrelevant?

    Per RGB paper: uses ONLY the 'negative' (irrelevant) documents from en.json.
    Since none of the documents contain the answer, the LLM should refuse.
    """
    results = []
    samples = data[:max_samples] if max_samples else data

    print(f'\n🚫 Negative Rejection — {model_name} — {len(samples)} samples')
    for i, item in enumerate(tqdm(samples, desc='Negative Rejection')):
        question = extract_question(item)

        # FIXED: Use only negative (irrelevant) documents
        neg_docs = get_negative_rejection_docs(item)

        if not question or not neg_docs:
            continue

        context = build_document_context(neg_docs)
        user_prompt = USER_PROMPT_REJECTION.format(context=context, question=question)

        response = call_llm(model_name, SYSTEM_PROMPT_REJECTION, user_prompt)
        rejected = check_rejection(response)

        results.append({
            'index': i,
            'question': question[:100],
            'response': response[:300] if response else 'FAILED',
            'rejected': rejected,
            'num_neg_docs': len(neg_docs),
        })
        time.sleep(delay)

    rejection_rate = sum(r['rejected'] for r in results) / len(results) if results else 0
    print(f'  ✅ Rejection Rate: {rejection_rate:.4f} ({sum(r["rejected"] for r in results)}/{len(results)})')
    return results, rejection_rate

print('✅ evaluate_negative_rejection defined')

In [ ]:
def evaluate_information_integration(model_name, data, max_samples=None, delay=1.5):
    """
    FIXED: Evaluate Information Integration: Can the LLM combine info from multiple docs?
    Uses en_int.json — questions that require synthesizing multiple documents.
    
    Key fixes:
    - Uses get_integration_docs() to flatten nested positive doc lists
    - Uses check_integration_accuracy() to verify ALL answer parts are found
    """
    results = []
    samples = data[:max_samples] if max_samples else data

    print(f'\n🔗 Information Integration — {model_name} — {len(samples)} samples')
    for i, item in enumerate(tqdm(samples, desc='Info Integration')):
        question = extract_question(item)
        gt_answers = extract_answer(item)
        documents = get_integration_docs(item)  # FIXED: flatten nested positive + negative

        if not question or not gt_answers or not documents:
            continue

        context = build_document_context(documents)
        user_prompt = USER_PROMPT_ACCURACY.format(context=context, question=question)

        response = call_llm(model_name, SYSTEM_PROMPT_ACCURACY, user_prompt)
        correct = check_integration_accuracy(response, gt_answers)  # FIXED: check ALL parts

        results.append({
            'index': i,
            'question': question[:100],
            'gt_answer': str(gt_answers),
            'response': response[:300] if response else 'FAILED',
            'correct': correct,
            'num_docs': len(documents),
        })
        time.sleep(delay)

    accuracy = sum(r['correct'] for r in results) / len(results) if results else 0
    print(f'  ✅ Accuracy: {accuracy:.4f} ({sum(r["correct"] for r in results)}/{len(results)})')
    return results, accuracy

print('✅ evaluate_information_integration defined')

In [ ]:
def evaluate_counterfactual_robustness(model_name, data, max_samples=None, delay=1.5):
    """
    FIXED: Evaluate Counterfactual Robustness: Can the LLM detect and correct factual errors?
    Uses en_fact.json — uses 'positive_wrong' documents that contain deliberately wrong facts.

    Returns TWO metrics:
      - Error Detection Rate: Did the LLM flag the error?
      - Error Correction Rate: Did the LLM still give the correct answer?
    """
    results = []
    samples = data[:max_samples] if max_samples else data

    print(f'\n🔍 Counterfactual Robustness — {model_name} — {len(samples)} samples')
    for i, item in enumerate(tqdm(samples, desc='Counterfactual')):
        question = extract_question(item)
        gt_answers = extract_answer(item)
        documents = get_counterfactual_docs(item)  # FIXED: use positive_wrong

        if not question or not gt_answers or not documents:
            continue

        context = build_document_context(documents)
        user_prompt = USER_PROMPT_COUNTERFACTUAL.format(context=context, question=question)

        response = call_llm(model_name, SYSTEM_PROMPT_COUNTERFACTUAL, user_prompt)
        detected = check_error_detection(response)
        corrected = check_error_correction(response, gt_answers)

        results.append({
            'index': i,
            'question': question[:100],
            'gt_answer': str(gt_answers),
            'fake_answer': str(item.get('fakeanswer', 'N/A')),
            'response': response[:300] if response else 'FAILED',
            'error_detected': detected,
            'error_corrected': corrected,
            'num_docs': len(documents),
        })
        time.sleep(delay)

    detection_rate = sum(r['error_detected'] for r in results) / len(results) if results else 0
    correction_rate = sum(r['error_corrected'] for r in results) / len(results) if results else 0
    print(f'  ✅ Error Detection Rate:  {detection_rate:.4f} ({sum(r["error_detected"] for r in results)}/{len(results)})')
    print(f'  ✅ Error Correction Rate: {correction_rate:.4f} ({sum(r["error_corrected"] for r in results)}/{len(results)})')
    return results, detection_rate, correction_rate

print('✅ evaluate_counterfactual_robustness defined')

## Step 6: Run the Full Evaluation
### 6a. Configuration
Set how many samples to evaluate per ability. Start small (20-30) to test, then scale up.

In [ ]:
# ============================================================
# EVALUATION CONFIGURATION
# ============================================================
# Start small (20-30) to test. Increase to None for full dataset.
MAX_SAMPLES_PER_ABILITY = 30   # Set to None for full evaluation
DELAY_BETWEEN_CALLS = 2.0       # Seconds between API calls (increase if rate limited)

print(f'Will evaluate {MAX_SAMPLES_PER_ABILITY or "ALL"} samples per ability')
print(f'Models to evaluate: {MODELS_TO_EVALUATE}')
print(f'Delay between calls: {DELAY_BETWEEN_CALLS}s')
print(f'\nEstimated time per model (30 samples × 4 abilities × {DELAY_BETWEEN_CALLS}s): ~{30 * 4 * DELAY_BETWEEN_CALLS / 60:.0f} min')

### 6b. Run Evaluation for All Models
This is the main execution cell. It will loop through each model and all 4 abilities.

In [ ]:
# ============================================================
# MAIN EVALUATION LOOP
# ============================================================
all_results = {}  # model_name -> {ability -> results}
summary_rows = []

for model_name in MODELS_TO_EVALUATE:
    print(f'\n{"="*60}')
    print(f'EVALUATING MODEL: {model_name}')
    print(f'{"="*60}')

    model_results = {}

    # 1. Noise Robustness (en.json, positive + negative docs)
    nr_results, nr_accuracy = evaluate_noise_robustness(
        model_name, data_en, MAX_SAMPLES_PER_ABILITY, DELAY_BETWEEN_CALLS
    )
    model_results['noise_robustness'] = nr_results

    # 2. Negative Rejection (en.json, negative docs only)
    rej_results, rej_rate = evaluate_negative_rejection(
        model_name, data_en, MAX_SAMPLES_PER_ABILITY, DELAY_BETWEEN_CALLS
    )
    model_results['negative_rejection'] = rej_results

    # 3. Information Integration (en_int.json, flattened positive + negative)
    ii_results, ii_accuracy = evaluate_information_integration(
        model_name, data_int, MAX_SAMPLES_PER_ABILITY, DELAY_BETWEEN_CALLS
    )
    model_results['information_integration'] = ii_results

    # 4. Counterfactual Robustness (en_fact.json, positive_wrong docs)
    cf_results, cf_detection, cf_correction = evaluate_counterfactual_robustness(
        model_name, data_fact, MAX_SAMPLES_PER_ABILITY, DELAY_BETWEEN_CALLS
    )
    model_results['counterfactual_robustness'] = cf_results

    all_results[model_name] = model_results

    # Collect summary
    summary_rows.append({
        'Model': model_name,
        'Noise Robustness (Accuracy)': f'{nr_accuracy:.4f}',
        'Negative Rejection (Rate)': f'{rej_rate:.4f}',
        'Info Integration (Accuracy)': f'{ii_accuracy:.4f}',
        'Counterfactual Detection': f'{cf_detection:.4f}',
        'Counterfactual Correction': f'{cf_correction:.4f}',
    })

print(f'\n{"="*60}')
print('ALL EVALUATIONS COMPLETE!')
print(f'{"="*60}')

## Step 7: Results Summary & Visualization
### 7a. Summary Table

In [ ]:
# Display summary table
df_summary = pd.DataFrame(summary_rows)
print('\n📊 RGB BENCHMARK RESULTS SUMMARY')
print('=' * 80)
display(df_summary)

# Save to CSV
df_summary.to_csv('rgb_results_summary.csv', index=False)
print('\n✅ Results saved to rgb_results_summary.csv')

### 7b. Visualization: Bar Chart Comparison

In [ ]:
# Prepare data for plotting
plot_data = []
for row in summary_rows:
    model = row['Model']
    plot_data.append({'Model': model, 'Ability': 'Noise Robustness', 'Score': float(row['Noise Robustness (Accuracy)'])})
    plot_data.append({'Model': model, 'Ability': 'Negative Rejection', 'Score': float(row['Negative Rejection (Rate)'])})
    plot_data.append({'Model': model, 'Ability': 'Info Integration', 'Score': float(row['Info Integration (Accuracy)'])})
    plot_data.append({'Model': model, 'Ability': 'Error Detection', 'Score': float(row['Counterfactual Detection'])})
    plot_data.append({'Model': model, 'Ability': 'Error Correction', 'Score': float(row['Counterfactual Correction'])})

df_plot = pd.DataFrame(plot_data)

# Create grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=df_plot, x='Ability', y='Score', hue='Model', ax=ax)
ax.set_title('RGB Benchmark: Model Comparison Across All Abilities', fontsize=14, fontweight='bold')
ax.set_ylabel('Score (0-1)', fontsize=12)
ax.set_xlabel('')
ax.set_ylim(0, 1.1)
ax.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8)

plt.tight_layout()
plt.savefig('rgb_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved to rgb_comparison_chart.png')

In [ ]:
# Individual subplot charts per ability
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

abilities = [
    ('Noise Robustness', 'Noise Robustness (Accuracy)'),
    ('Negative Rejection', 'Negative Rejection (Rate)'),
    ('Info Integration', 'Info Integration (Accuracy)'),
    ('Error Detection', 'Counterfactual Detection'),
]

colors = sns.color_palette('husl', len(MODELS_TO_EVALUATE))

for ax, (title, col) in zip(axes, abilities):
    scores = [float(row[col]) for row in summary_rows]
    models_short = [m.split('/')[-1][:15] for m in MODELS_TO_EVALUATE]
    bars = ax.bar(models_short, scores, color=colors)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{score:.2f}', ha='center', fontsize=9)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('RGB Benchmark: Per-Ability Model Scores', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('rgb_per_ability_charts.png', dpi=150, bbox_inches='tight')
plt.show()

### 7c. Detailed Results: Sample-Level Analysis

In [ ]:
# Show sample-level results for debugging and analysis
for model_name in MODELS_TO_EVALUATE:
    print(f'\n{"="*60}')
    print(f'MODEL: {model_name}')
    print(f'{"="*60}')

    # Noise Robustness samples
    nr = all_results[model_name]['noise_robustness']
    if nr:
        df_nr = pd.DataFrame(nr)
        print(f'\n--- Noise Robustness (first 5 incorrect) ---')
        incorrect = df_nr[df_nr['correct'] == 0].head(5)
        if len(incorrect) > 0:
            for _, row in incorrect.iterrows():
                print(f'  Q: {row["question"]}')
                print(f'  GT: {row["gt_answer"]}')
                print(f'  Response: {row["response"][:150]}')
                print()

    # Negative Rejection samples
    rej = all_results[model_name]['negative_rejection']
    if rej:
        df_rej = pd.DataFrame(rej)
        print(f'--- Negative Rejection (first 3 where LLM did NOT reject) ---')
        not_rejected = df_rej[df_rej['rejected'] == 0].head(3)
        if len(not_rejected) > 0:
            for _, row in not_rejected.iterrows():
                print(f'  Q: {row["question"]}')
                print(f'  Response: {row["response"][:200]}')
                print()
        else:
            print(f'  All samples correctly rejected! 🎉')

    # Counterfactual samples
    cf = all_results[model_name]['counterfactual_robustness']
    if cf:
        df_cf = pd.DataFrame(cf)
        print(f'--- Counterfactual (first 3 where error NOT detected) ---')
        not_detected = df_cf[df_cf['error_detected'] == 0].head(3)
        if len(not_detected) > 0:
            for _, row in not_detected.iterrows():
                print(f'  Q: {row["question"]}')
                print(f'  Fake answer: {row.get("fake_answer", "N/A")}')
                print(f'  Response: {row["response"][:200]}')
                print()

### 7d. Save All Results to JSON

In [ ]:
# Save detailed results for later analysis
import json

output = {
    'config': {
        'models': MODELS_TO_EVALUATE,
        'max_samples': MAX_SAMPLES_PER_ABILITY,
        'delay': DELAY_BETWEEN_CALLS,
    },
    'summary': summary_rows,
    'detailed_results': {
        model: {
            ability: results
            for ability, results in model_results.items()
        }
        for model, model_results in all_results.items()
    }
}

with open('rgb_full_results.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)

print('✅ Full results saved to rgb_full_results.json')
print('✅ Summary saved to rgb_results_summary.csv')
print('✅ Charts saved to rgb_comparison_chart.png and rgb_per_ability_charts.png')

## Step 8 (BONUS): Improve LLM Abilities Without Changing the Model
The RGB bonus asks: can you improve the 4 abilities using **prompt engineering only**?

Ideas to try:
1. **Chain-of-Thought (CoT):** Add "Think step by step" to prompts
2. **Few-Shot Examples:** Add 2-3 example Q&A pairs before the actual question
3. **Explicit Instructions:** More detailed instructions for rejection/detection
4. **Self-Consistency:** Sample multiple responses and take majority vote

Below is a template for CoT prompting on Noise Robustness:

In [ ]:
# BONUS: Chain-of-Thought prompt for improved Noise Robustness
SYSTEM_PROMPT_COT = """You are a helpful assistant. Answer the question based on the given documents.
Think step by step:
1. First, identify which documents are relevant to the question.
2. Ignore any documents that are not related to the question.
3. Extract the specific answer from the relevant documents.
4. Provide a concise and direct final answer."""

# To evaluate: replace SYSTEM_PROMPT_ACCURACY with SYSTEM_PROMPT_COT
# in evaluate_noise_robustness() and compare results.

# Example: Run CoT on 10 samples of noise robustness with model 0
# Uncomment below to test:
# print('Testing CoT prompt...')
# cot_results, cot_accuracy = evaluate_noise_robustness(
#     MODELS_TO_EVALUATE[0], data_en, max_samples=10, delay=2.0
# )
# print(f'CoT Accuracy: {cot_accuracy:.4f}')